## 16. Employee Intelligence Table

Everything from the last two days lands here - **one row per employee** pulling together:
- Attrition risk (NB05-NB07)
- Engagement (NB10)
- Role (NB11)
- Skill gaps (NB13-NB14)
- Training recommendation (NB15)

**This table is the actual business output of the whole project. The dashboard is basically just a view onto it.**

```
Employee_ID | Dept | Attrition_Prob | Risk | Engagement | Role | Skill_Gap | Recommendation
101         | IT   | 0.81           | HIGH | 62         | Data Analyst | MLOps,Docker | Learn MLOps
```

In [1]:
import pandas as pd
import numpy as np
import os
import ast
import joblib
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

PROCESSED_PATH = '../data/processed'
MODELS_PATH = '../models'
DOCS_PATH = '../docs'

# Load all data sources
emp = pd.read_csv(f'{PROCESSED_PATH}/employee_attrition_processed.csv')
engagement = pd.read_csv(f'{PROCESSED_PATH}/engagement_processed.csv')
emp_skills = pd.read_csv(f'{PROCESSED_PATH}/employee_skills.csv')
emp_gaps = pd.read_csv(f'{PROCESSED_PATH}/skill_gaps.csv')
recs = pd.read_csv(f'{PROCESSED_PATH}/employee_recommendations.csv')
pipeline = joblib.load(f'{MODELS_PATH}/attrition_pipeline.joblib')

print(f'Employees: {emp.shape[0]}')
print(f'Engagement records: {engagement.shape[0]}')
print(f'Recommendations: {recs.shape[0]}')

Employees: 500
Engagement records: 3000
Recommendations: 4983


## 1. Attrition Risk Prediction

Load the trained pipeline and predict attrition probability for all employees.

In [2]:
# Load features from NB05 (includes engineered features)
X_features = pd.read_csv(f'{PROCESSED_PATH}/../features/X_features.csv')

# Load column metadata
with open(f'{PROCESSED_PATH}/../features/column_metadata.json') as f:
    metadata = json.load(f)

numeric_cols = metadata['numeric_cols']
categorical_cols = metadata['categorical_cols']
feature_cols = numeric_cols + categorical_cols

# Use X_features for prediction (has all engineered features)
X = X_features[feature_cols].copy()

# Predict probabilities
y_proba = pipeline.predict_proba(X)[:, 1]
emp['Attrition_Prob'] = y_proba.round(4)

# Assign risk levels
def assign_risk(prob):
    if prob >= 0.6:
        return 'HIGH'
    elif prob >= 0.3:
        return 'MEDIUM'
    else:
        return 'LOW'

emp['Risk'] = emp['Attrition_Prob'].apply(assign_risk)

print('Risk Distribution:')
print(emp['Risk'].value_counts().to_string())
print()
print(f"High-risk employees: {len(emp[emp['Risk'] == 'HIGH'])}")

Risk Distribution:
Risk
LOW       265
MEDIUM    152
HIGH       83

High-risk employees: 83


## 2. Engagement Score

Average engagement score per employee from NB10.

In [3]:
# Average engagement score per employee
# NOTE: Employee IDs (1-500) and Engagement IDs (1001-4000) don't overlap (documented in NB04/NB10)
# Using Department-level averages as fallback

eng_by_dept = engagement.groupby('BusinessUnit').agg({
    'Satisfaction Score': 'mean',
    'Work-Life Balance Score': 'mean',
    'Engagement Score': 'mean'
}).reset_index()

# Map engagement BusinessUnit to employee Department (approximate)
# Since IDs don't overlap, use department averages from engagement as proxy
eng_dept_map = {}
for _, row in eng_by_dept.iterrows():
    eng_dept_map[row['BusinessUnit']] = {
        'Satisfaction': row['Satisfaction Score'],
        'WorkLifeBalance': row['Work-Life Balance Score'],
        'Engagement': row['Engagement Score']
    }

# Assign engagement scores based on department averages
import numpy as np
emp['Satisfaction'] = np.nan
emp['WorkLifeBalance'] = np.nan
emp['Engagement_Score'] = np.nan

# Use overall engagement averages as proxy (since IDs don't overlap)
overall_sat = engagement['Satisfaction Score'].mean()
overall_wlb = engagement['Work-Life Balance Score'].mean()
overall_eng = engagement['Engagement Score'].mean()

emp['Satisfaction'] = overall_sat
emp['WorkLifeBalance'] = overall_wlb
emp['Engagement_Score'] = overall_eng

print(f'NOTE: Employee IDs (1-500) and Engagement IDs (1001-4000) do not overlap')
print(f'Using overall engagement averages as proxy')
print(f'Avg Satisfaction: {overall_sat:.2f}')
print(f'Avg Work-Life Balance: {overall_wlb:.2f}')
print(f'Avg Engagement: {overall_eng:.2f}')

NOTE: Employee IDs (1-500) and Engagement IDs (1001-4000) do not overlap
Using overall engagement averages as proxy
Avg Satisfaction: 3.02
Avg Work-Life Balance: 2.99
Avg Engagement: 2.94


## 3. Skill Gap Summary

Top skill gaps per employee from NB13/NB14.

In [4]:
# Parse skill gaps from NB13 output
# emp_gaps is department-level. Let's recalculate employee-level gaps.
role_master = pd.read_csv(f'{PROCESSED_PATH}/role_master.csv')

def safe_parse_list(val):
    if isinstance(val, str) and val.startswith('['):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return val if isinstance(val, list) else []

dept_keywords = {
    'IT': ['software', 'computer', 'network', 'database', 'systems', 'developer', 'it'],
    'Sales': ['sales', 'marketing', 'advertising', 'representative', 'account'],
    'Support': ['customer', 'support', 'service', 'help', 'desk'],
    'Finance': ['finance', 'accounting', 'financial', 'analyst', 'manager'],
    'HR': ['human', 'resources', 'hr', 'training', 'development'],
    'Marketing': ['marketing', 'public', 'relations', 'communications']
}

def map_to_department(title):
    if pd.isna(title):
        return 'Other'
    title_lower = str(title).lower()
    for dept, keywords in dept_keywords.items():
        if any(kw in title_lower for kw in keywords):
            return dept
    return 'Other'

role_master['Mapped_Department'] = role_master['Title'].apply(map_to_department)

dept_required = role_master.groupby('Mapped_Department').agg({
    'Essential_Skills': lambda x: set(
        item for sublist in x for item in safe_parse_list(sublist)
    ),
    'Software_Skills': lambda x: set(
        item for sublist in x for item in safe_parse_list(sublist)
    )
}).reset_index()
dept_required['All_Required'] = dept_required.apply(
    lambda row: row['Essential_Skills'].union(row['Software_Skills']), axis=1
)
dept_lookup = dict(zip(dept_required['Mapped_Department'], dept_required['All_Required']))

# Build employee skill sets
emp_skill_sets = emp_skills.groupby('EmployeeID')['current_skill'].apply(set).reset_index()
emp_skill_sets.columns = ['EmployeeID', 'Current_Skills']
emp_skill_sets = emp_skill_sets.merge(emp[['EmployeeID', 'Department']], on='EmployeeID')

# Calculate gaps
gap_data = []
for _, row in emp_skill_sets.iterrows():
    dept = row['Department']
    current = row['Current_Skills']
    required = dept_lookup.get(dept, set())
    gap = required - current
    gap_data.append({
        'EmployeeID': row['EmployeeID'],
        'Skill_Gap': list(gap)[:5],  # Top 5 gaps
        'Gap_Count': len(gap)
    })

gap_df = pd.DataFrame(gap_data)
emp = emp.merge(gap_df, on='EmployeeID', how='left')
emp['Skill_Gap'] = emp['Skill_Gap'].apply(lambda x: x if isinstance(x, list) else [])
emp['Gap_Count'] = emp['Gap_Count'].fillna(0).astype(int)

print(f'Average skill gap: {emp["Gap_Count"].mean():.0f} skills')

Average skill gap: 969 skills


## 4. Training Recommendation

Top recommendation per employee from NB15.

In [5]:
# Get top recommendation per employee
top_recs = recs.groupby('EmployeeID').first().reset_index()
top_recs = top_recs[['EmployeeID', 'Recommended_Course', 'Duration']]
top_recs.columns = ['EmployeeID', 'Top_Recommendation', 'Training_Duration']

emp = emp.merge(top_recs, on='EmployeeID', how='left')

print(f'Employees with recommendations: {top_recs.shape[0]}')
print(f'Top recommendations:')
print(emp['Top_Recommendation'].value_counts().head(10).to_string())

Employees with recommendations: 500
Top recommendations:
Top_Recommendation
PowerPoint Presentations          330
Git Version Control               132
Google Analytics Certification     22
Advanced Excel                     16


## 5. Build Final Intelligence Table

One row per employee with all metrics.

In [6]:
# Create the intelligence table
intelligence = emp[[
    'EmployeeID', 'Department', 'JobRole', 'Age', 'Gender',
    'YearsAtCompany', 'MonthlySalary', 'PerformanceRating',
    'Attrition_Prob', 'Risk',
    'Satisfaction', 'WorkLifeBalance', 'Engagement_Score',
    'Gap_Count', 'Skill_Gap', 'Top_Recommendation', 'Training_Duration'
]].copy()

# Format skill gaps as comma-separated string
intelligence['Skill_Gap_Str'] = intelligence['Skill_Gap'].apply(
    lambda x: ', '.join(x[:3]) if x else 'None'
)

# Round numeric columns
intelligence['Attrition_Prob'] = intelligence['Attrition_Prob'].round(4)
intelligence['Satisfaction'] = intelligence['Satisfaction'].round(2)
intelligence['WorkLifeBalance'] = intelligence['WorkLifeBalance'].round(2)
intelligence['Engagement_Score'] = intelligence['Engagement_Score'].round(2)

print(f'Intelligence table: {intelligence.shape}')
print(f'Columns: {intelligence.columns.tolist()}')

Intelligence table: (500, 18)
Columns: ['EmployeeID', 'Department', 'JobRole', 'Age', 'Gender', 'YearsAtCompany', 'MonthlySalary', 'PerformanceRating', 'Attrition_Prob', 'Risk', 'Satisfaction', 'WorkLifeBalance', 'Engagement_Score', 'Gap_Count', 'Skill_Gap', 'Top_Recommendation', 'Training_Duration', 'Skill_Gap_Str']


## 6. Preview

In [7]:
print('First 10 rows (key columns):')
display_cols = ['EmployeeID', 'Department', 'Attrition_Prob', 'Risk',
                'Engagement_Score', 'Gap_Count', 'Skill_Gap_Str', 'Top_Recommendation']
print(intelligence[display_cols].head(10).to_string(index=False))

First 10 rows (key columns):
 EmployeeID Department  Attrition_Prob   Risk  Engagement_Score  Gap_Count                                                                                        Skill_Gap_Str       Top_Recommendation
          1    Finance          0.5334 MEDIUM              2.94       1433        Microsoft Power BI, Brihzon Solutions SMART Dairy Resource Planning, TechHackers Swap @nalyst      Git Version Control
          2      Sales          0.7897   HIGH              2.94        670               Astute Solutions PowerCenter, MI-Assistant MI Management System, General Magic Portico PowerPoint Presentations
          3    Support          0.1067    LOW              2.94        399 Delivery operations information system DOIS, Electronic medical record EMR software, StataCorp Stata PowerPoint Presentations
          4         HR          0.2436    LOW              2.94        397       Bond International Software Bond StaffSuite, SAP ERP Human Capital Management, Google 

## 7. Risk Segments

In [8]:
print('Risk Segment Analysis:')
for risk in ['HIGH', 'MEDIUM', 'LOW']:
    segment = intelligence[intelligence['Risk'] == risk]
    if len(segment) > 0:
        print(f"\n{risk} Risk ({len(segment)} employees):")
        print(f"  Avg Attrition Prob: {segment['Attrition_Prob'].mean():.2%}")
        print(f"  Avg Engagement: {segment['Engagement_Score'].mean():.2f}")
        print(f"  Avg Skill Gap: {segment['Gap_Count'].mean():.0f}")
        print(f"  Top Departments: {segment['Department'].value_counts().head(3).to_dict()}")

Risk Segment Analysis:

HIGH Risk (83 employees):
  Avg Attrition Prob: 71.33%
  Avg Engagement: 2.94
  Avg Skill Gap: 1257
  Top Departments: {'IT': 24, 'HR': 19, 'Finance': 17}

MEDIUM Risk (152 employees):
  Avg Attrition Prob: 46.76%
  Avg Engagement: 2.94
  Avg Skill Gap: 883
  Top Departments: {'Sales': 33, 'Marketing': 28, 'Finance': 26}

LOW Risk (265 employees):
  Avg Attrition Prob: 10.95%
  Avg Engagement: 2.94
  Avg Skill Gap: 928
  Top Departments: {'Support': 52, 'Sales': 51, 'IT': 48}


## 8. Dashboard-Ready View

In [9]:
# Create a clean dashboard-ready view
dashboard_view = intelligence[[
    'EmployeeID', 'Department', 'JobRole', 'Risk',
    'Attrition_Prob', 'Engagement_Score', 'Satisfaction',
    'Gap_Count', 'Skill_Gap_Str', 'Top_Recommendation'
]].copy()

dashboard_view.columns = [
    'Employee_ID', 'Department', 'Role', 'Risk_Level',
    'Attrition_Probability', 'Engagement_Score', 'Satisfaction_Score',
    'Skill_Gap_Count', 'Top_Skill_Gaps', 'Recommended_Training'
]

print('Dashboard-ready view:')
print(dashboard_view.head(10).to_string(index=False))

Dashboard-ready view:
 Employee_ID Department            Role Risk_Level  Attrition_Probability  Engagement_Score  Satisfaction_Score  Skill_Gap_Count                                                                                       Top_Skill_Gaps     Recommended_Training
           1    Finance         Auditor     MEDIUM                 0.5334              2.94                3.02             1433        Microsoft Power BI, Brihzon Solutions SMART Dairy Resource Planning, TechHackers Swap @nalyst      Git Version Control
           2      Sales Sales Executive       HIGH                 0.7897              2.94                3.02              670               Astute Solutions PowerCenter, MI-Assistant MI Management System, General Magic Portico PowerPoint Presentations
           3    Support        Helpdesk        LOW                 0.1067              2.94                3.02              399 Delivery operations information system DOIS, Electronic medical record EMR software,

## 9. Save Intelligence Table

In [10]:
# Save full intelligence table
intelligence.to_csv(f'{PROCESSED_PATH}/employee_intelligence.csv', index=False)

# Save dashboard view
dashboard_view.to_csv(f'{PROCESSED_PATH}/dashboard_data.csv', index=False)

# Save as JSON for dashboard
dashboard_view.to_json(f'{PROCESSED_PATH}/dashboard_data.json', orient='records', indent=2)

print(f'Saved employee_intelligence.csv: {intelligence.shape}')
print(f'Saved dashboard_data.csv: {dashboard_view.shape}')
print(f'Saved dashboard_data.json')

Saved employee_intelligence.csv: (500, 18)
Saved dashboard_data.csv: (500, 10)
Saved dashboard_data.json


## Summary

### What This Table Contains

| Column | Source | Description |
|--------|--------|-------------|
| Attrition_Prob | NB05-NB07 | Predicted probability of leaving |
| Risk | NB07 | HIGH/MEDIUM/LOW risk level |
| Engagement_Score | NB10 | Average engagement rating |
| Satisfaction | NB10 | Average satisfaction rating |
| WorkLifeBalance | NB10 | Average work-life balance rating |
| Gap_Count | NB13-NB14 | Number of missing skills |
| Skill_Gap | NB13 | Top missing skills |
| Top_Recommendation | NB15 | Recommended training course |

### Business Value

- **One row per employee** - complete view of HR intelligence
- **Dashboard-ready** - `dashboard_data.csv` is the direct feed
- **Actionable** - each employee has a clear next step (training recommendation)

### Files Created

- `data/processed/employee_intelligence.csv` - full intelligence table
- `data/processed/dashboard_data.csv` - clean dashboard view
- `data/processed/dashboard_data.json` - JSON format for web dashboards